# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from typing import Literal
import time

from rich.pretty import pprint

import numpy as np

from enderscope.scan_patterns import snake, plot_path
from enderscope.serial import list_ports, Stage
from enderscope.bed import bed
from enderscope.enderlights import Enderlights

import enderleaf.acquire as ea
import enderleaf.image as ei
from enderleaf.tools import time_method

In [ ]:
%matplotlib widget

## Constants

In [ ]:
TEMPLATE_LENGTH = 210
TEMPLATE_SIZE = (TEMPLATE_LENGTH, TEMPLATE_LENGTH)
ROW_COUNT, COL_COUNT = 9, 9
LEAF_DIAM = 17
CAM_RES = (4608, 2592)

## Scan Pattern

In [ ]:
positions = snake(cols=COL_COUNT, rows=ROW_COUNT) * [
    # steps
    TEMPLATE_LENGTH / COL_COUNT,
    TEMPLATE_LENGTH / ROW_COUNT,
] + [
    # origin
    TEMPLATE_LENGTH / COL_COUNT / 2,
    TEMPLATE_LENGTH / ROW_COUNT / 2,
]
plot_path(
    positions,
    title="snake scan",
    field=(LEAF_DIAM + 3, LEAF_DIAM + 3),
    selected_rectangles=[17, 41, 55, 81],
    circle_diam=LEAF_DIAM,
)

## 3D Virtual Scan

In [ ]:
s = Stage("virtual", 115200)

In [ ]:
s.home()

In [ ]:
s.move_position((bed.x_max / 2, bed.y_max / 2, bed.global_height))
for p in positions:
    s.move_position(np.append(p, bed.individual_height))

## 3D Scan

In [ ]:
def dummy_acquire_image(
    stage: Stage,
    lights: Enderlights,
    pos,
    read_qr: bool,
    resolution: tuple,
    focus_mode: Literal[ea.FocusMode.MANUAL, ea.FocusMode.HUNT, ea.FocusMode.AUTO],
    crop_data: ei.Rectangle | None = None,
):
    stage.move_position(pos)
    stage.finish_moves()
    lights.shutter(True)
    ea.capture(resulution=resolution, focus_mode=focus_mode, crop_data=crop_data)
    time.sleep(1)
    lights.shutter(False)


#    if read_qr is True:
#        print("QR read")

In [ ]:
# list available serial ports
stage_port = None
light_port = None
ports = list_ports()
for port in ports:
    if "USB Serial" in port.description:
        stage_port = port
        print("* " + str(port))
    elif "UART" in port.description:
        light_port = port
        print("+ " + str(port))
    else:
        print("  " + str(port))

In [ ]:
stage = Stage(stage_port, 115200)
lights = Enderlights(light_port, 57600)

stage.home()
stage.finish_moves()

In [ ]:
@time_method
def run_job(stage, lights, speed):
    s.write_code(f"M203 Z{speed}")
    dummy_acquire_image(
        stage=stage,
        lights=lights,
        pos=(bed.x_max / 2, bed.y_max / 2, bed.global_height),
        read_qr=True,
        resolution=CAM_RES,
        focus_mode=ea.FocusMode.AUTO,
    )

    for i, p in enumerate(positions):
        dummy_acquire_image(
            stage=stage,
            lights=lights,
            pos=np.append(p, bed.individual_height),
            read_qr=i == 0,
            resolution=CAM_RES,
            focus_mode=ea.FocusMode.HUNT,
            crop_data=ei.Rectangle(top=0, left=0, right=CAM_RES[0], bottom=CAM_RES[1])
            .shrink(new_height=1000, new_width=1000)
            .ensure_int(),
        )


run_job(stage, lights, 50)
run_job(stage, lights, 5)